# Dataset generators

This project builds synthetic sequence-pair datasets through a small registry of
**generators**, living in `dnabind.datagen`. A generator is a function that returns a
`pandas.DataFrame` with the `Seq1`, `Seq2`, `Label` columns that
`dnabind.data.PairDataset` consumes (plus any extra descriptive columns).

The registry mirrors the `encoders` and `models` packages: drop a new module in
`src/dnabind/datagen/` that decorates a function with `@register_generator("name")`,
and it is discovered automatically — no wiring needed.

## Discover what's available

`list_generators()` returns every registered generator; `get_generator(name)` hands
back the callable.

In [1]:
from dnabind.datagen import get_generator, list_generators

list_generators()

['complementary', 'offset']

## The `offset` generator

`offset` builds a controlled **offset-ladder**: a random 20-mer backbone (`Seq1`) is
paired with a partner (`Seq2`) so that `Seq1` and `RC(Seq2)` form one contiguous
Watson-Crick duplex shifted by a signed `offset` d. A shift of `|d|` leaves `|d|`
overhang bases, so the bound length is `L = 20 - |d|`. The **same** backbones are
reused at every offset, so each backbone appears across the whole ladder — supporting
paired, per-backbone analysis downstream.

Every construct is verified with a parasail local alignment (exact `L`-bp block on the
intended diagonal, no stray off-diagonal complementarity), so the realised duplex is
exactly the one requested.

Signature: `generate_offset_dataset(n_backbones, offsets, seed, stray_threshold=5, max_tries=200)`.

In [2]:
gen = get_generator("offset")

# A small ladder: perfect duplex (0), plus +/-5 and +/-10 shifts.
df = gen(n_backbones=2, offsets=[0, 5, -5, 10, -10], seed=0)
df.head()

,backbone_id,offset,bind_length,gc_bound,Seq1,Seq2,Label
0,0,0,20,0.5500,TTAGTTGTGCCGCAGCGAAG,CTTCGCTGCGGCACAACTAA,1
1,0,5,15,0.6667,TTAGTTGTGCCGCAGCGAAG,CACTACTTCGCTGCGGCACA,1
2,0,-5,15,0.5333,TTAGTTGTGCCGCAGCGAAG,CTGCGGCACAACTAATCAAG,1
3,0,10,10,0.7000,TTAGTTGTGCCGCAGCGAAG,GTCGCATATTCTTCGCTGCG,1
4,0,-10,10,0.4000,TTAGTTGTGCCGCAGCGAAG,GCACAACTAAGCATACGCTC,1


## Save to a CSV for training

The `Seq1, Seq2, Label` columns are all `PairDataset` reads; the rest are descriptive.
Write it anywhere under `data/` and point the `dnabind` CLI at it.

In [ ]:
from pathlib import Path

out = Path("../data/offset_ladder/offset_ladder.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f"wrote {len(df)} rows -> {out}")

For larger runs from the shell, the same generator is exposed as a module CLI:

```bash
python -m dnabind.datagen.offset \
    --n-backbones 200 --min-offset 5 --max-offset 10 \
    --include-zero --seed 0 --out data/offset_ladder/offset_ladder.csv
```

## Adding a new generator

Create `src/dnabind/datagen/<your_dataset>.py`:

```python
import pandas as pd
from .registry import register_generator

@register_generator("your_dataset")
def generate_your_dataset(...) -> pd.DataFrame:
    ...  # return a DataFrame with Seq1, Seq2, Label
```

It will show up in `list_generators()` automatically.